# Baomai: test the reviewed visible-front extents

**Draft: the published source commit must be pinned before running.** Once published, select a CUDA GPU, Run all, and return one ZIP. The implementation is pinned to `d43a58b9456a782ace14e20f0bf7ed2da3572ca1`; the exact candidate and protocol are verified before training. This is one matched label comparison.

Both arms start independently from original epoch 173 with the unchanged seed 46 recipe: 30 epochs, batch 4, image size 1280, FP32, AdamW, learning rate 0.0001 and bias warmup 0.0001. Select finalized epoch 30 `last.pt`.

| Outer role | Training labels | Inner arm |
| --- | --- | --- |
| `old_reviewed` | Exact `annotations-2026-11-10` eight-file overlay | `candidate` |
| `extended_reviewed` | Same eight files plus the frozen accepted visible-extent cohort delta | `candidate` |

The six-image review scope is 061/062/063/079/082/094. Only accepted changes enter the explicit protocol delta; reviewed uncertain cases can remain unchanged. The source split, original validation labels, initialization, recipe and runtime are identical between arms. No resource-trial model or previously trained candidate initializes either arm.

After return, primary inference confidence is 0.13; 0.15 is the only historical diagnostic. Image size 1280, NMS 0.70, endpoint cutoff 0.50 and the corrected 663-object evaluation reference stay fixed. This notebook does no inference, promotion or accuracy reporting.

Completed six-artifact bundles and the final ZIP are backed up to Drive. Matching completed arms are restored and skipped; partial runs remain ineligible. Original-image review and mechanical label verification do not establish learned-model improvement.


## 1. Select a GPU and prepare the workspace

In Colab choose **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. Both full runs must use the same GPU/runtime and pinned implementation. The first cell refuses unfilled source/protocol pins before GPU or filesystem actions. Approve the Drive mount so completed runs survive loss of `/content`.

This notebook can reopen its own matching workspace. It refuses a workspace or Drive artifact belonging to different pins. If an interrupted arm remains locally, preserve it and use a fresh `RUN_BASE`; completed Drive bundles will be restored. A different GPU/runtime cannot be silently mixed with a completed arm.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time

SOURCE_REVISION = "d43a58b9456a782ace14e20f0bf7ed2da3572ca1"
PROTOCOL_PATH = "experiments/visible-extent-2026-11-11.json"
PROTOCOL_SHA256 = "42aa275a5d2ed1ccabb786c07b537a8f0d515b27b7804fec2b71e942d6c9022b"
RECIPE_ID = "reviewed-annotations-seed46-v1"
RECIPE_SHA256 = "f18e0a813a1057e1b3ab3a44ea3ce9b2a3e7673cc2bf04ca09b394bfdd8c51b8"
RUN_BASE = Path("/content/baomai-visible-extent-seed46")
ROLES = ("old_reviewed", "extended_reviewed")

assert re.fullmatch(r"[0-9a-f]{40}", SOURCE_REVISION), "Use the final handoff notebook with its full source commit pinned."
assert re.fullmatch(r"[0-9a-f]{64}", PROTOCOL_SHA256), "Use the final handoff notebook with its protocol hash pinned."
assert sys.version_info >= (3, 11), "Python3.11 or newer is required."

def run(command, *, cwd=None, env=None):
    print("+", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)

def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

run([sys.executable, "-c", "import torch, torchvision; "
     "assert torch.cuda.is_available(), 'Select a CUDA GPU runtime before continuing.'; "
     "print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda); "
     "print('GPU:', torch.cuda.get_device_name(0)); "
     "print('CUDA calculation:', (torch.ones(1, device='cuda:0') * 2).item())"])
run(["nvidia-smi"])
workspace_identity = {"source_revision": SOURCE_REVISION, "protocol_sha256": PROTOCOL_SHA256}
if RUN_BASE.exists() or RUN_BASE.is_symlink():
    assert RUN_BASE.is_dir() and not RUN_BASE.is_symlink(), "Choose a fresh nonsymlink workspace."
    marker = RUN_BASE / "workspace-identity.json"
    assert marker.is_file() and json.loads(marker.read_text()) == workspace_identity, "Existing workspace belongs to different pins; preserve it and choose a new RUN_BASE."
else:
    RUN_BASE.mkdir(parents=True)
    (RUN_BASE / "workspace-identity.json").write_text(json.dumps(workspace_identity, indent=2) + "\n")
REPO = RUN_BASE / "dl"
DRIVE_ROOT = None
try:
    from google.colab import drive
except ImportError:
    print("Drive is unavailable here; preserve completed bundles and the ZIP from the local workspace.")
else:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/baomai-visible-extent-seed46-20261111")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive folder:", DRIVE_ROOT)

def safe_member(name):
    p = Path(name)
    assert isinstance(name, str) and name and not p.is_absolute() and ".." not in p.parts
    assert "\\" not in name and ":" not in name and p.as_posix() == name
    return p

def artifact_inventory(path):
    path = Path(path)
    if path.is_file():
        names = [path.name]
        base = path.parent
    else:
        base = path
        receipt = json.loads((path / "training-receipt.json").read_text())
        names = ["training-receipt.json", "staging-manifest.json", receipt["log"]["path"]]
        if (path / "checkpoint-manifest.json").is_file():
            manifest = json.loads((path / "checkpoint-manifest.json").read_text())
            names += ["checkpoint-manifest.json", manifest["checkpoint"]["path"], receipt["results"]["path"]]
    assert len(names) == len(set(names))
    return [{"path": name, "sha256": sha256(base / safe_member(name)), "size_bytes": (base / name).stat().st_size}
            for name in sorted(names)]

def validate_copy(path, marker):
    base = path if marker["directory"] else path.parent
    for entry in marker["files"]:
        target = base / safe_member(entry["path"])
        assert target.is_file() and not target.is_symlink(), f"Missing backup artifact: {target}"
        assert target.stat().st_size == entry["size_bytes"] and sha256(target) == entry["sha256"], f"Backup hash mismatch: {target}"

def save_to_drive(path):
    if DRIVE_ROOT is None:
        return
    path = Path(path)
    relative = path.relative_to(RUN_BASE).as_posix()
    if relative == "resource-trial":
        relative = "resource-trials/" + sha256(path / "training-receipt.json")[:20]
    destination = DRIVE_ROOT / safe_member(relative)
    marker_path = Path(str(destination) + ".complete")
    marker = {**workspace_identity, "relative_path": relative, "directory": path.is_dir(), "files": artifact_inventory(path)}
    if marker_path.is_file():
        assert json.loads(marker_path.read_text()) == marker, f"Drive already has different completed evidence: {destination}"
        validate_copy(destination, marker)
        print("Verified existing Drive backup:", destination)
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Only an incomplete copy in this notebook's exact destination is replaced.
    if destination.is_dir():
        shutil.rmtree(destination)
    elif destination.exists():
        destination.unlink()
    if path.is_dir():
        destination.mkdir()
        for entry in marker["files"]:
            target = destination / safe_member(entry["path"])
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path / entry["path"], target)
    else:
        shutil.copyfile(path, destination)
    validate_copy(destination, marker)
    marker_path.write_text(json.dumps(marker, indent=2) + "\n")
    print("Saved verified completed artifacts:", destination, flush=True)

def restore_from_drive(relative):
    if DRIVE_ROOT is None:
        return None
    source = DRIVE_ROOT / safe_member(relative)
    marker_path = Path(str(source) + ".complete")
    if not marker_path.is_file():
        return None
    marker = json.loads(marker_path.read_text())
    assert all(marker.get(k) == v for k, v in workspace_identity.items()), "Drive completion belongs to different source/protocol pins."
    assert marker["relative_path"] == relative
    validate_copy(source, marker)
    destination = RUN_BASE / safe_member(relative)
    if destination.exists() or destination.is_symlink():
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    if marker["directory"]:
        destination.mkdir()
        for entry in marker["files"]:
            target = destination / safe_member(entry["path"])
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source / entry["path"], target)
    else:
        assert len(marker["files"]) == 1 and marker["files"][0]["path"] == source.name
        shutil.copyfile(source, destination)
    validate_copy(destination, marker)
    print("Restored completed evidence:", destination, flush=True)
    return destination

print("Workspace:", RUN_BASE)


## 2. Download and verify both exact label variants

Selective Git LFS fetches the 157 original PNGs and epoch 173 weights. The existing annotation verifier checks both overlays, full row mappings, preserved corner tokens and unchanged validation. The new helper also checks that all eight old corrected files remain byte-identical and only the declared accepted image IDs differ. Nothing is trained in this cell.


In [ ]:
lfs = subprocess.run(["git", "lfs", "version"], capture_output=True, text=True)
if lfs.returncode:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "git-lfs"])
git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0"}
if not REPO.exists():
    run(["git", "init", REPO], env=git_env)
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], cwd=REPO)
    run(["git", "fetch", "--depth=1", "origin", SOURCE_REVISION], cwd=REPO, env=git_env)
    run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO, env=git_env)
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_revision == SOURCE_REVISION
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
assert sha256(REPO / PROTOCOL_PATH) == PROTOCOL_SHA256
run(["git", "lfs", "install", "--local"], cwd=REPO)
reference_manifest = json.loads((REPO / "data/manifest.json").read_text())
expected_images = {"data/" + row["image"] for row in reference_manifest["records"]}
tracked_images = set(subprocess.check_output(
    ["git", "ls-files", "data/images/train/*.png", "data/images/val/*.png"], cwd=REPO, text=True).splitlines())
assert len(expected_images) == 157 and tracked_images == expected_images
assert reference_manifest["model_checkpoint"] == "weights/cassette_lower_epoch173.pt"
assert reference_manifest["model_epoch"] == 173
run(["git", "lfs", "pull", "--include=data/images/train/*.png,data/images/val/*.png,weights/cassette_lower_epoch173.pt", "--exclude="], cwd=REPO)
input_preflight = r"""
import json
from pathlib import Path
import sys
sys.dont_write_bytecode = True
root = Path.cwd()
sys.path.insert(0, str(root / 'scripts'))
from evaluate_pose import preflight_dataset
from stage_pose_training import source_state
from package_visible_extent_results import validate_protocol
context = validate_protocol(root, sys.argv[1])
assert context['sha256'] == sys.argv[2]
initial = preflight_dataset(root, require_weights=True)
counts = {}
for role, spec in context['document']['roles'].items():
    state = source_state(root, root / spec['candidate_manifest'])
    assert len(state['records']) == 157
    assert state['source']['initial_checkpoint_sha256'] == initial['model_sha256']
    assert state['counts']['reference']['val'] == state['counts']['candidate']['val']
    counts[role] = state['counts']['candidate']
print(json.dumps({'status':'verified', 'original_images':157, 'label_counts':counts,
                  'weights_verified':initial['weights_verified'],
                  'protocol_sha256':context['sha256'],
                  'delta_train_image_ids':context['delta_train_image_ids']}, indent=2))
"""
run([sys.executable, "-B", "-c", input_preflight, PROTOCOL_PATH, PROTOCOL_SHA256], cwd=REPO)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
print("Verified clean pinned source:", actual_revision)


## 3. Preserve the CUDA runtime and install the pinned trainer

Torch and torchvision stay constrained to the GPU runtime's installed versions. A dependency conflict stops the workflow; it does not trigger a silent CPU fallback or runtime substitution. Both full runs use the same environment and single GPU.


In [ ]:
from importlib.metadata import version
constraints = RUN_BASE / "torch-constraints.txt"
constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("torch", "torchvision")), encoding="utf-8")
run([sys.executable, "-m", "pip", "install", "-c", constraints, "ultralytics==8.4.182", "psutil"])
run([sys.executable, "-c", "import torch, torchvision, ultralytics; "
     "assert ultralytics.__version__ == '8.4.182'; assert torch.cuda.is_available(); "
     "print('Ready:', torch.__version__, torchvision.__version__, ultralytics.__version__, torch.cuda.get_device_name(0))"])
import psutil
HOST_RSS_LIMIT_MIB = min(12000, int(psutil.virtual_memory().available / 2**20 * 0.80))
assert HOST_RSS_LIMIT_MIB >= 6000, "Free more host RAM or choose a larger-memory runtime before training."
print("Host-RAM stop budget (not GPU VRAM):", HOST_RSS_LIMIT_MIB, "MiB")
print("Limits:600 seconds for the trial;21600 seconds for each complete run.")


## 4. Verify recovery and run a short trial when needed

A four-batch real training trial uses the extended labels and the unchanged 30-epoch schedule, then stops without an eligible checkpoint. Both full runs initialize independently from epoch 173. Completed arms are verified before any new GPU training; recovered arms, the trial and the current runtime must match.

If both arms are complete, skip the resource trial and all training. Otherwise, reuse a complete compatible saved trial when available; corrupt or incompatible trial copies stay preserved and are skipped. A partial local run is rejected. Use a new workspace after an interrupted run; completed Drive bundles can then be restored without copying staged images.


In [ ]:
sys.dont_write_bytecode = True
if str(REPO / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO / "scripts"))
from evaluate_pose import preflight_dataset
from package_visible_extent_results import validate_protocol, collect_reviewed_run, check_reviewed_pair
from pose_checkpoint import recipe_identity
from train_pose_comparison import source_identity
CONTEXT = validate_protocol(REPO, PROTOCOL_PATH)
PREFLIGHT = preflight_dataset(REPO, require_weights=False)
assert CONTEXT["sha256"] == PROTOCOL_SHA256
assert recipe_identity(RECIPE_ID)["recipe_sha256"] == RECIPE_SHA256


def assert_finished(output, role, *, trial=False):
    output = Path(output)
    if not trial:
        bundle = collect_reviewed_run(PREFLIGHT, output, role, CONTEXT)
        assert bundle["receipt"]["source"] == source_identity(REPO), "Completed arm comes from a different implementation."
        return bundle["receipt"]
    receipt = json.loads((output / "training-receipt.json").read_text())
    staging = json.loads((output / "staging-manifest.json").read_text())
    spec = CONTEXT["document"]["roles"][role]
    assert receipt["status"] == "trial-complete" and receipt["purpose"] == "resource-trial"
    assert receipt["arm"] == "candidate" and receipt["source"] == source_identity(REPO)
    assert all(receipt[k] == v for k, v in recipe_identity(RECIPE_ID).items())
    assert all(staging["source"][k] == v for k, v in spec.items())
    assert receipt["initial_checkpoint_sha256"] == CONTEXT["document"]["initial_checkpoint_sha256"]
    assert receipt["inputs_reverified_after_worker"] is True
    assert receipt["execution"]["exit_code"] == 0 and receipt["execution"]["stop_reason"] is None
    assert receipt["optimizer_steps"] > 0 and receipt["optimizer_parameter_change_observed"] is True
    assert receipt["runtime"]["device"].startswith("cuda:")
    assert receipt["staging_manifest_sha256"] == sha256(output / "staging-manifest.json")
    assert not (output / "checkpoint-manifest.json").exists()
    return receipt


def assert_current_runtime(receipt, current, label):
    expected = receipt["runtime"]
    assert all(expected[k] == v for k, v in current.items() if k != "cudnn_version"), f"Current runtime differs from {label}; select a matching runtime before full training."
    assert expected["backend"]["cudnn_version"] == current["cudnn_version"], f"Current cuDNN runtime differs from {label}; no full arm started."


def train_one(role, *, trial=False):
    assert role in ROLES
    folder = "resource-trial" if trial else role
    output = RUN_BASE / folder
    if output.exists() or output.is_symlink():
        try:
            receipt = assert_finished(output, role, trial=trial)
        except Exception as error:
            raise FileExistsError(f"Preserve incomplete or incompatible run {output}. Use a fresh RUN_BASE for retry; completed Drive arms will be restored. {error}") from error
        print("Skipping verified completed run:", folder, flush=True)
        save_to_drive(output)
        return output, receipt
    command = [sys.executable, "-B", "scripts/train_pose_comparison.py",
               "--trial" if trial else "--train", "--arm", "candidate", "--recipe", RECIPE_ID,
               "--candidate", CONTEXT["document"]["roles"][role]["candidate_manifest"],
               "--device", "0", "--threads", "4", "--trial-batches", "4",
               "--max-seconds", "600" if trial else "21600",
               "--max-rss-mib", str(HOST_RSS_LIMIT_MIB), "--output", str(output)]
    print("Starting:", folder, role, RECIPE_ID, flush=True)
    print("Log:", output / "train.log", flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, start_new_session=True)
    descendants = {}
    def remember_descendants():
        try:
            for child in psutil.Process(process.pid).children(recursive=True):
                descendants[child.pid] = child
        except psutil.NoSuchProcess:
            pass
    started, last_notice = time.monotonic(), -30.0
    try:
        while process.poll() is None:
            remember_descendants()
            elapsed = time.monotonic() - started
            if elapsed - last_notice >= 20:
                progress_file = output / "training-progress.json"
                if progress_file.exists():
                    progress = json.loads(progress_file.read_text())
                    print(f"{folder}:{elapsed:.0f}s; status={progress.get('status')}; epochs={len(progress.get('completed_epochs', []))}/30; batches={progress.get('batches_completed', 0)}; updates={progress.get('optimizer_steps', 0)}", flush=True)
                else:
                    print(f"{folder}:{elapsed:.0f}s; verifying and staging inputs", flush=True)
                last_notice = elapsed
            time.sleep(1)
        captured, _ = process.communicate()
        print(captured)
        if process.returncode:
            log = output / "train.log"
            if log.exists():
                print("\n".join(log.read_text(errors="replace").splitlines()[-45:]))
            raise RuntimeError(f"{folder} failed; preserve its receipt/log and use a fresh workspace for retry.")
    except BaseException:
        remember_descendants()
        children = list(descendants.values())
        for child in reversed(children):
            try:
                child.terminate()
            except psutil.NoSuchProcess:
                pass
        if process.poll() is None:
            process.terminate()
        _, alive = psutil.wait_procs(children, timeout=5)
        for child in alive:
            try:
                child.kill()
            except psutil.NoSuchProcess:
                pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    receipt = assert_finished(output, role, trial=trial)
    save_to_drive(output)
    return output, receipt


def recover_trial(current):
    local = RUN_BASE / "resource-trial"
    if local.exists() or local.is_symlink():
        return train_one("extended_reviewed", trial=True)
    if DRIVE_ROOT is not None:
        markers = sorted((DRIVE_ROOT / "resource-trials").glob("*.complete"),
                         key=lambda path: path.stat().st_mtime, reverse=True)
        for marker in markers:
            relative = marker.relative_to(DRIVE_ROOT).as_posix().removesuffix(".complete")
            try:
                restored = restore_from_drive(relative)
                receipt = assert_finished(restored, "extended_reviewed", trial=True)
                assert_current_runtime(receipt, current, "resource trial")
                for completed in COMPLETED.values():
                    assert receipt["runtime"] == completed["runtime"], "Saved trial differs from a completed arm."
            except (AssertionError, ValueError, OSError, KeyError, TypeError) as error:
                print("Skipped incompatible saved trial:", relative, str(error))
                continue
            print("Reused verified completed trial:", restored)
            return restored, receipt
    return train_one("extended_reviewed", trial=True)


COMPLETED = {}
for role in ROLES:
    restore_from_drive(role)
    output = RUN_BASE / role
    if output.exists() or output.is_symlink():
        COMPLETED[role] = assert_finished(output, role)
        print("Recovered eligible completed arm:", role)
TRIAL_RUNTIME = None
if len(COMPLETED) == 2:
    check_reviewed_pair({role: collect_reviewed_run(PREFLIGHT, RUN_BASE / role, role, CONTEXT) for role in ROLES}, CONTEXT)
    print("Both arms are complete; no resource trial or training will repeat.")
else:
    probe = r"""
import importlib.metadata as m, json, platform, torch
assert torch.cuda.is_available()
print(json.dumps({'python':platform.python_version(),'torch':torch.__version__,
'ultralytics':m.version('ultralytics'),'torch_cuda':torch.version.cuda,'platform':platform.platform(),
'device':'cuda:0','device_name':torch.cuda.get_device_name(0),'device_capability':list(torch.cuda.get_device_capability(0)),
'packages':{n:m.version(n) for n in ('torchvision','numpy','Pillow','opencv-python','psutil')},
'cudnn_version':torch.backends.cudnn.version()}))
"""
    current = json.loads(subprocess.check_output([sys.executable, "-c", probe], text=True, cwd=REPO))
    for role, receipt in COMPLETED.items():
        assert_current_runtime(receipt, current, f"completed {role}")
    TRIAL, trial_receipt = recover_trial(current)
    assert_current_runtime(trial_receipt, current, "resource trial")
    TRIAL_RUNTIME = trial_receipt["runtime"]
    for role, receipt in COMPLETED.items():
        assert receipt["runtime"] == TRIAL_RUNTIME, f"Actual trial runtime differs from completed {role}; no full arm started."
    print("Resource trial passed; it is not a completed model.")


## 5. Train only the missing matched arms

The old-reviewed control runs first, followed by the extended-reviewed candidate. Both invoke the unchanged runner with `--arm candidate`; only their explicitly pinned annotation manifests differ. Verified completed runs are skipped. Final epoch 30 is fixed in advance; no best-epoch or seed search occurs.


In [ ]:
OLD_REVIEWED, old_receipt = train_one("old_reviewed")
if TRIAL_RUNTIME is not None:
    assert old_receipt["runtime"] == TRIAL_RUNTIME, "Old-reviewed runtime differs from the trial; do not start the other arm."
EXTENDED_REVIEWED, extended_receipt = train_one("extended_reviewed")
assert old_receipt["runtime"] == extended_receipt["runtime"], "Completed arms have different runtimes; do not report a matched result."
for role, folder in (("old_reviewed", OLD_REVIEWED), ("extended_reviewed", EXTENDED_REVIEWED)):
    receipt = assert_finished(folder, role)
    print(role, "inner arm:", receipt["arm"], "final checkpoint:", receipt["checkpoint"]["sha256"])
print("Both matched arms completed. Package the unchanged artifacts next.")


## 6. Verify and package one return ZIP

The separate packager validates both completed candidate bundles, equal source/runtime/settings, epoch 173 initialization, preserved old corrected labels and the exact declared four-image delta. Original validation images and labels must be identical. The ZIP contains six artifacts per role plus its comparison index. Receipts are never relabeled as `reference`.

An existing completed ZIP is reverified without overwriting. No weights are deserialized by packaging.


In [ ]:
RESULTS_ZIP = RUN_BASE / "baomai-visible-extent-seed46-results.zip"
restore_from_drive(RESULTS_ZIP.name)
command = [sys.executable, "-B", "scripts/package_visible_extent_results.py",
           "--protocol", PROTOCOL_PATH, "--old-reviewed", OLD_REVIEWED,
           "--extended-reviewed", EXTENDED_REVIEWED, "--output", RESULTS_ZIP]
if RESULTS_ZIP.exists():
    command.append("--verify-existing")
run(command, cwd=REPO)
print("Return:", RESULTS_ZIP.name)
print("Bytes:", RESULTS_ZIP.stat().st_size)
print("SHA-256:", sha256(RESULTS_ZIP))
save_to_drive(RESULTS_ZIP)


## 7. Download and return the ZIP

Attach **`baomai-visible-extent-seed46-results.zip`** in chat or return it through the agreed results PR. It is also backed up to the notebook's Google Drive folder. Keep the two completed bundles and their logs until the result is reviewed.


In [ ]:
assert RESULTS_ZIP.is_file(), "Complete the packaging cell first."
from google.colab import files
files.download(str(RESULTS_ZIP))


### After the return

Return `baomai-visible-extent-seed46-results.zip`. Keep the verified Drive copies until the comparison is complete. Evaluation must retain both references separately, the current PR35 operational comparator, every gained/lost object and all visibility/corner/scene tradeoffs. A gain on disputed reference rows alone is not recovery of a physically supported target.

The old-reviewed control is a freshly trained candidate-label arm. Do not replace it with original labels, initialize from PR35, or use terminal `best.pt` metrics in place of the selected `last.pt` evaluation. No further training is authorized by the notebook's completion itself.
